In [ ]:

from mmdet.registry import VISUALIZERS
import sys
from pathlib import Path
import torch 

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')


from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose

from cam import EigenCAM
import rasterio as rio 
import numpy as np

def estimate_vmin_vmax(cam, percentiles=[5, 95]):
    """
    Estimates vmin and vmax values for the heatmap.

    Parameters:
    - cam (numpy.ndarray): The heatmap.

    Returns:
    - tuple: A tuple containing the vmin and vmax values.
    """
    vmin = np.percentile(cam, percentiles[0])
    vmax = np.percentile(cam, percentiles[1])
    return vmin, vmax

# Required by loader
def read_tif(file_path, band_indices):
    """
    Reads specified bands from a TIFF file.

    Parameters:
    - file_path (str): Path to the .tif file.
    - band_indices (list of int): Indices of the bands to read.

    Returns:
    - numpy.ndarray: A numpy array containing the stacked band data.
    """
    data = []
    with rio.open(file_path) as src:
        for index in band_indices:
            data.append(src.read(index))
    stacked_data = np.stack(data, axis=0)
    return np.transpose(stacked_data, (1, 2, 0))

DATA_PATH_VEN = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
DATA_PATH_SEN = '/Data_large/marine/Datasets/VDS2Raw/imgs'

TIFF_VEN = list(Path(DATA_PATH_VEN).rglob('*.tif'))
TIFF_SEN = list(Path(DATA_PATH_SEN).rglob('*.tif'))


# Specify the path to model config and checkpoint file
config_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/vfnet_r18.py'
checkpoint_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()

Idx = 157
tiffSel = None
test_pipeline = Compose(test_pipeline)
if tiffSel is None:
    data_ = dict(img_path=TIFF_VEN[Idx], img_id=0)
    ORIGINAL_IMG = read_tif(file_path=TIFF_VEN[Idx], band_indices=[5])

else:
    data_ = dict(img_path=tiffSel, img_id=0)
    ORIGINAL_IMG = read_tif(file_path=tiffSel, band_indices=[5])
    

data_ = test_pipeline(data_)
data_['inputs'] = [data_['inputs']]
data_['data_samples'] = [data_['data_samples']]


# forward the model
with torch.no_grad():
    results = model.test_step(data_)[0]


In [ ]:
image_height = 2048
image_width = 2048
dummy_input = torch.randn(1, 1, image_height, image_width).to(device)  # Ensure dummy input is on the same device as the model

with torch.no_grad():
    output = model(dummy_input)

for o in output:
    for subo in o:
        print(subo.shape)

In [ ]:
# from torchsummary import summary

# # Print the summary of the model's backbone
# summary(model, input_size=(1, 2048, 2048))


In [ ]:
x = data_['inputs']
inp = x[0].to(device).unsqueeze(0)
inp.shape

Forward pass and export to onnx

In [ ]:
# Forward pass
with torch.no_grad():
    outputs = model(inp)


# Export the model to ONNX format
torch.onnx.export(model.backbone, inp, 'model_encoder.onnx', opset_version=11)
inp_neck = list(model.backbone(inp))
torch.onnx.export(model.neck, inp_neck, 'model_neck.onnx', opset_version=11)

In [ ]:
out_neck = model.neck(inp_neck)

torch.onnx.export(model.bbox_head, list(out_neck), 'model_heads.onnx', opset_version=11)





In [ ]:
for o in outputs[0]:
    print(o.shape)

In [ ]:
results

In [1]:
import mmdet 
from mmdet.models.dense_heads import VFNetHead, FCOSHead
import torch 
import warnings
warnings.filterwarnings("ignore")

# testare teste di mmdet: 
INF = 1e8
testa = VFNetHead(11, 7)
testa = FCOSHead(11, 7)

# dummy input:
feats = [torch.rand(1, 7, s, s) for s in [4, 8, 16, 32, 64]]
cls_score, bbox_pred, bbox_pred_refine= testa.forward(feats)

assert len(cls_score) == len(testa.scales)


# export to onnx testa
dummy_input = [torch.randn(1, 7, s, s) for s in [4, 8, 16, 32, 64]]
torch.onnx.export(testa, dummy_input, "testa.onnx", verbose=True, input_names=["input"], output_names=["cls_score", "bbox_pred", "bbox_pred_refine"])
